# BÀI THI / ÔN TẬP THỰC HÀNH GIỮA KỲ MÔN HỌC MÁY (MACHINE LEARNING)
**Họ và tên sinh viên:** Nguyễn Đỗ Thắng  
**MSSV:** 102230046  
**Lớp:** 23T_NHAT1  

---
### NỘI DUNG YÊU CẦU ĐỀ THI:
1. **Phân đoạn ảnh (Image Segmentation):** Phân cụm ảnh màu (RGB) và ảnh trắng đen (Grayscale) bằng thuật toán **K-Means** và **Fuzzy C-Means (FCM)** (tối thiểu 4-5 cụm, thử nghiệm 5 và 7 cụm).
2. **Dữ liệu bảng (Dataset):** Thực hiện phân cụm bằng **K-Means**, **FCM** và phân lớp bằng **K-NN**.
3. **Yêu cầu kỹ thuật bắt buộc:**
   - **Bắt buộc tự code thuật toán (Custom from scratch)** bằng NumPy, không được gọi hàm thuật toán từ thư viện để làm chính.
   - **So sánh với thư viện có sẵn (Scikit-Learn, Scikit-Fuzzy):** So sánh kết quả, thời gian, biểu đồ đối sánh side-by-side và bảng biểu số liệu rõ ràng.
   - **Biểu đồ chứng minh lý do chọn $K$ và chọn $C$:**
     - Với K-Means: Phương pháp **Elbow Method (WCSS/Inertia)** và **Silhouette Score**.
     - Với FCM: Hệ số phân hoạch mờ **FPC (Fuzzy Partition Coefficient)** và **Silhouette Score**.
     - Với K-NN: Biểu đồ **Tỷ lệ lỗi (Error Rate)** và **Độ chính xác (Accuracy)** theo các giá trị $K$ lân cận.
   - **Phân biệt bản chất:** Sự khác biệt cốt lõi giữa xử lý ảnh màu và ảnh trắng đen trong phân đoạn ảnh.


In [ ]:
# 1. CÀI ĐẶT THƯ VIỆN & CẤU HÌNH HIỂN THỊ
import time
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Thư viện scikit-learn để làm đối chứng so sánh
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN
from sklearn.metrics import silhouette_score, accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cấu hình đồ họa đẹp mắt, hiển thị trực tiếp trong notebook
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110

print("Đã nạp đầy đủ các thư viện cần thiết!")


---
# PHẦN 1: PHÂN ĐOẠN ẢNH (IMAGE SEGMENTATION) BẰNG K-MEANS & FUZZY C-MEANS

### 1.1. Phân biệt bản chất giữa Ảnh màu (RGB) và Ảnh trắng đen (Grayscale) trong Phân cụm
Trong bài toán phân đoạn ảnh bằng học máy:
1. **Cấu trúc dữ liệu & Số chiều không gian đặc trưng:**
   - **Ảnh màu (RGB):** Là tensor 3 chiều kích thước $H \times W \times 3$. Mỗi điểm ảnh (pixel) là một vector đặc trưng 3 chiều biểu thị cường độ màu:
     $$\vec{p} = [R, G, B]^T \in [0, 255]^3$$
     Khi đưa vào thuật toán phân cụm, ảnh được làm phẳng (flatten/reshape) thành ma trận dữ liệu:
     $$X_{color} \in \mathbb{R}^{(H \cdot W) \times 3}$$
     Các cụm tìm được đại diện cho **các nhóm màu chủ đạo (color palettes)** trong ảnh.
   - **Ảnh trắng đen / Mức xám (Grayscale):** Là ma trận 2 chiều kích thước $H \times W$. Mỗi điểm ảnh chỉ có 1 kênh duy nhất biểu thị cường độ sáng (luminance/intensity):
     $$p = I \in [0, 255]$$
     Khi đưa vào phân cụm, ảnh được reshape thành ma trận 1 chiều:
     $$X_{gray} \in \mathbb{R}^{(H \cdot W) \times 1}$$
     Các cụm tìm được phân vùng ảnh theo **các mức độ sáng tối (độ chói / contrast)**, giúp tách biệt các vùng sáng, vùng xám trung gian và vùng tối/bóng râm.

2. **Cách tính khoảng cách:**
   - Với ảnh màu: Khoảng cách Euclidean trong không gian 3D:
     $$d(p_1, p_2) = \sqrt{(R_1 - R_2)^2 + (G_1 - G_2)^2 + (B_1 - B_2)^2}$$
   - Với ảnh xám: Khoảng cách là độ chênh lệch cường độ sáng 1D:
     $$d(p_1, p_2) = |I_1 - I_2|$$

3. **Tái tạo ảnh sau phân cụm:**
   Mỗi điểm ảnh được thay thế bằng giá trị màu của tâm cụm (centroid) mà nó thuộc về, giúp giảm số màu trong ảnh về đúng $K$ màu đại diện.


In [ ]:
# 1.2. CÀI ĐẶT THUẬT TOÁN K-MEANS TỪ ĐẦU (CUSTOM K-MEANS FROM SCRATCH)
class CustomKMeans:
    """
    Thuật toán K-Means phân cụm tự cài đặt 100% bằng NumPy thuần.
    - Hỗ trợ khoảng cách: 'euclidean' hoặc 'manhattan'.
    - Tối ưu hóa vector hóa (vectorization) để xử lý ảnh hàng vạn pixel cực nhanh.
    """
    def __init__(self, k=5, max_iters=100, tol=1e-4, metric='euclidean', random_state=42):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.metric = metric
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None  # WCSS: Tổng bình phương khoảng cách tới tâm cụm
        self.n_iter_ = 0

    def _compute_distance(self, X, centroids):
        """Tính ma trận khoảng cách giữa các điểm X (N, D) và các tâm (K, D)"""
        if self.metric == 'euclidean':
            return np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        elif self.metric == 'manhattan':
            return np.sum(np.abs(X[:, np.newaxis, :] - centroids[np.newaxis, :, :]), axis=2)
        else:
            raise ValueError("Metric phải là 'euclidean' hoặc 'manhattan'")

    def _init_centroids(self, X):
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        # Khởi tạo ngẫu nhiên k điểm trong X làm tâm ban đầu
        indices = np.random.choice(n_samples, self.k, replace=False)
        return X[indices].astype(float)

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]
        self.centroids = self._init_centroids(X)

        for it in range(self.max_iters):
            self.n_iter_ = it + 1
            # Bước 1: Gán mỗi điểm vào tâm cụm gần nhất
            dists = self._compute_distance(X, self.centroids)
            labels = np.argmin(dists, axis=1)

            # Bước 2: Cập nhật lại tâm cụm bằng trung bình các điểm thuộc cụm
            new_centroids = np.zeros_like(self.centroids)
            for j in range(self.k):
                members = X[labels == j]
                if len(members) > 0:
                    new_centroids[j] = np.mean(members, axis=0)
                else:
                    new_centroids[j] = X[np.random.choice(n_samples)]

            # Kiểm tra điều kiện hội tụ
            shift = np.linalg.norm(new_centroids - self.centroids)
            self.centroids = new_centroids
            if shift < self.tol:
                break

        # Gán nhãn lần cuối và tính Inertia (WCSS)
        dists = self._compute_distance(X, self.centroids)
        self.labels_ = np.argmin(dists, axis=1)
        euc_dists = np.linalg.norm(X - self.centroids[self.labels_], axis=1)
        self.inertia_ = float(np.sum(euc_dists ** 2))
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        dists = self._compute_distance(X, self.centroids)
        return np.argmin(dists, axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

print("Đã định nghĩa class CustomKMeans thành công!")


In [ ]:
# 1.3. CÀI ĐẶT THUẬT TOÁN FUZZY C-MEANS (FCM) TỪ ĐẦU
class CustomFCM:
    """
    Thuật toán Phân cụm mờ Fuzzy C-Means (FCM) tự cài đặt 100% bằng NumPy.
    - c: Số lượng cụm
    - m: Fuzziness exponent (hệ số mờ, thường m = 2.0)
    - tol: Ngưỡng dừng biến thiên ma trận thuộc tính U
    """
    def __init__(self, c=5, m=2.0, max_iters=100, tol=1e-4, random_state=42):
        self.c = c
        self.m = m
        self.max_iters = max_iters
        self.tol = tol
        self.random_state = random_state
        self.centers = None
        self.u = None           # Ma trận mức độ thuộc U kích thước (N, c)
        self.labels_ = None     # Nhãn cứng bằng argmax(U)
        self.obj_history = []
        self.n_iter_ = 0

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]

        # Khởi tạo ma trận U ngẫu nhiên sao cho tổng mỗi hàng bằng 1
        np.random.seed(self.random_state)
        self.u = np.random.dirichlet(np.ones(self.c), size=n_samples)

        for it in range(self.max_iters):
            self.n_iter_ = it + 1
            u_old = self.u.copy()

            # 1. Cập nhật tọa độ tâm cụm v_i:
            # v_i = sum_k (u_ik^m * x_k) / sum_k (u_ik^m)
            um = self.u ** self.m
            self.centers = (um.T @ X) / np.sum(um, axis=0)[:, np.newaxis]

            # 2. Tính ma trận khoảng cách d_ij = ||x_j - v_i||
            dists = np.linalg.norm(X[:, np.newaxis, :] - self.centers[np.newaxis, :, :], axis=2)
            dists = np.fmax(dists, 1e-10)

            # 3. Tính hàm mục tiêu J_m
            j_m = float(np.sum((self.u ** self.m) * (dists ** 2)))
            self.obj_history.append(j_m)

            # 4. Cập nhật ma trận mức độ thuộc u_ik:
            # u_ik = 1 / sum_j ( (d_ik / d_jk) ^ (2 / (m - 1)) )
            power = 2.0 / (self.m - 1.0)
            inv_dists = 1.0 / dists
            inv_dists_power = inv_dists ** power
            self.u = inv_dists_power / np.sum(inv_dists_power, axis=1, keepdims=True)

            # Kiểm tra điều kiện dừng
            if np.max(np.abs(self.u - u_old)) < self.tol:
                break

        self.labels_ = np.argmax(self.u, axis=1)
        return self

    def compute_fpc(self):
        """Fuzzy Partition Coefficient (FPC) nằm trong [1/c, 1], càng gần 1 phân hoạch càng rõ rệt"""
        return float(np.sum(self.u ** 2) / len(self.u))

print("Đã định nghĩa class CustomFCM thành công!")


In [ ]:
# 1.4. NẠP ẢNH ĐỂ THỰC HIỆN PHÂN ĐOẠN
# (LƯU Ý THI CỬ: Thầy phát ảnh nào, bạn chỉ cần dán đường dẫn ảnh đó vào biến 'IMAGE_PATH')

IMAGE_PATH = 'sample_color_image.jpg'

if not os.path.exists(IMAGE_PATH):
    # Nếu chưa có ảnh, tự động sinh bức ảnh màu chất lượng cao để test ngay
    h, w = 150, 200
    xx, yy = np.meshgrid(np.linspace(0, 1, w), np.linspace(0, 1, h))
    # Tạo các dải màu phong phú (hoa quả / cảnh sắc)
    r = np.clip(np.sin(xx * np.pi * 1.5) * 255 + 50, 0, 255)
    g = np.clip(np.cos(yy * np.pi * 1.2) * 255 + 60, 0, 255)
    b = np.clip(((xx - 0.5)**2 + (yy - 0.5)**2) * 600, 0, 255)
    synthetic_img = np.stack([r, g, b], axis=-1).astype(np.uint8)
    Image.fromarray(synthetic_img).save(IMAGE_PATH)
    print(f"Đã tạo ảnh màu mẫu tại: {IMAGE_PATH}")

# Tải ảnh gốc (RGB) và ảnh mức xám (Grayscale)
raw_img = Image.open(IMAGE_PATH)
raw_img.thumbnail((250, 250), Image.Resampling.LANCZOS) # Resize hợp lý để tính toán nhanh trong phòng thi

img_color_np = np.array(raw_img.convert('RGB'), dtype=float)
img_gray_np = np.array(raw_img.convert('L'), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(img_color_np.astype('uint8'))
axes[0].set_title(f"Ảnh màu gốc (RGB)\nKích thước: {img_color_np.shape[1]}x{img_color_np.shape[0]}x3", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_gray_np.astype('uint8'), cmap='gray')
axes[1].set_title(f"Ảnh trắng đen gốc (Grayscale)\nKích thước: {img_gray_np.shape[1]}x{img_gray_np.shape[0]}", fontweight='bold')
axes[1].axis('off')

plt.suptitle("HÌNH ẢNH ĐẦU VÀO ĐƯỢC CHUẨN BỊ CHO BÀI TOÁN PHÂN ĐOẠN", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 1.5. PHÂN ĐOẠN ẢNH MÀU BẰNG K-MEANS (THỬ NGHIỆM K=5 VÀ K=7)
# Yêu cầu đề thi: Tối thiểu 4-5 cụm, chọn 5 và 7 cụm, so sánh Custom vs Thư viện

h, w, c = img_color_np.shape
# Flatten ma trận (H, W, 3) thành (H*W, 3)
pixels_color = img_color_np.reshape(-1, 3)

results_color = []

for k in [5, 7]:
    print(f"\nĐang thực thi phân đoạn ảnh màu với K = {k}...")
    
    # 1. Custom K-Means
    t0 = time.time()
    custom_km = CustomKMeans(k=k, random_state=42, max_iters=50)
    custom_km.fit(pixels_color)
    t_custom = time.time() - t0
    custom_seg = custom_km.centroids[custom_km.labels_].reshape(h, w, 3).astype('uint8')
    
    # 2. Sklearn K-Means
    t0 = time.time()
    sk_km = SklearnKMeans(n_clusters=k, random_state=42, n_init=5, max_iter=50)
    sk_km.fit(pixels_color)
    t_sk = time.time() - t0
    sk_seg = sk_km.cluster_centers_[sk_km.labels_].reshape(h, w, 3).astype('uint8')
    
    # Lưu kết quả
    results_color.append({
        'K': k,
        'Custom Time (s)': t_custom,
        'Sklearn Time (s)': t_sk,
        'Custom WCSS': custom_km.inertia_,
        'Sklearn WCSS': sk_km.inertia_,
        'Custom Iterations': custom_km.n_iter_,
        'Sklearn Iterations': sk_km.n_iter_
    })
    
    # Vẽ biểu đồ so sánh 3 cột: Gốc | Custom | Thư viện
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(img_color_np.astype('uint8'))
    axes[0].set_title("Ảnh màu gốc", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(custom_seg)
    axes[1].set_title(f"Custom K-Means (K={k})\nThời gian: {t_custom:.3f}s | WCSS: {custom_km.inertia_:.1e}", fontweight='bold', color='navy')
    axes[1].axis('off')
    
    axes[2].imshow(sk_seg)
    axes[2].set_title(f"Sklearn K-Means (K={k})\nThời gian: {t_sk:.3f}s | WCSS: {sk_km.inertia_:.1e}", fontweight='bold', color='darkgreen')
    axes[2].axis('off')
    
    plt.suptitle(f"SO SÁNH PHÂN ĐOẠN ẢNH MÀU VỚI K = {k}", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Xuất bảng số liệu so sánh rõ ràng theo yêu cầu đề thi
df_color_comp = pd.DataFrame(results_color)
print("\n--- BẢNG SO SÁNH HIỆU NĂNG PHÂN ĐOẠN ẢNH MÀU ---")
display(df_color_comp)


In [ ]:
# 1.6. PHÂN ĐOẠN ẢNH TRẮNG ĐEN (GRAYSCALE) BẰNG K-MEANS (K=5 VÀ K=7)
# Ảnh xám chỉ có 1 kênh duy nhất: (H, W) -> reshape thành (H*W, 1)

pixels_gray = img_gray_np.reshape(-1, 1)
results_gray = []

for k in [5, 7]:
    print(f"\nĐang thực thi phân đoạn ảnh trắng đen với K = {k}...")
    
    # 1. Custom K-Means
    t0 = time.time()
    custom_km_gray = CustomKMeans(k=k, random_state=42, max_iters=50)
    custom_km_gray.fit(pixels_gray)
    t_custom = time.time() - t0
    custom_seg_gray = custom_km_gray.centroids[custom_km_gray.labels_].reshape(h, w).astype('uint8')
    
    # 2. Sklearn K-Means
    t0 = time.time()
    sk_km_gray = SklearnKMeans(n_clusters=k, random_state=42, n_init=5, max_iter=50)
    sk_km_gray.fit(pixels_gray)
    t_sk = time.time() - t0
    sk_seg_gray = sk_km_gray.cluster_centers_[sk_km_gray.labels_].reshape(h, w).astype('uint8')
    
    results_gray.append({
        'K': k,
        'Custom Time (s)': t_custom,
        'Sklearn Time (s)': t_sk,
        'Custom WCSS': custom_km_gray.inertia_,
        'Sklearn WCSS': sk_km_gray.inertia_
    })
    
    # Vẽ biểu đồ so sánh ảnh xám
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(img_gray_np.astype('uint8'), cmap='gray')
    axes[0].set_title("Ảnh xám gốc", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(custom_seg_gray, cmap='gray')
    axes[1].set_title(f"Custom K-Means Xám (K={k})\nThời gian: {t_custom:.3f}s | WCSS: {custom_km_gray.inertia_:.1e}", fontweight='bold', color='navy')
    axes[1].axis('off')
    
    axes[2].imshow(sk_seg_gray, cmap='gray')
    axes[2].set_title(f"Sklearn K-Means Xám (K={k})\nThời gian: {t_sk:.3f}s | WCSS: {sk_km_gray.inertia_:.1e}", fontweight='bold', color='darkgreen')
    axes[2].axis('off')
    
    plt.suptitle(f"SO SÁNH PHÂN ĐOẠN ẢNH TRẮNG ĐEN VỚI K = {k}", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

df_gray_comp = pd.DataFrame(results_gray)
print("\n--- BẢNG SO SÁNH HIỆU NĂNG PHÂN ĐOẠN ẢNH TRẮNG ĐEN ---")
display(df_gray_comp)


In [ ]:
# 1.7. PHÂN ĐOẠN ẢNH BẰNG FUZZY C-MEANS (FCM)
# Áp dụng thuật toán phân cụm mờ Custom FCM trên ảnh (chọn c = 5 cụm)

# Do FCM tính toán ma trận mờ U kích thước (N x c), ta lấy mẫu hoặc resize nhỏ để chạy nhanh
sample_step = 2 # Lấy mẫu pixel bước 2 để thuật toán chạy mượt trong 1-2 giây
sampled_pixels = pixels_color[::sample_step]

print(f"Đang chạy Custom Fuzzy C-Means (c = 5) trên {len(sampled_pixels)} điểm ảnh mẫu...")
t0 = time.time()
fcm_model = CustomFCM(c=5, m=2.0, max_iters=40, tol=1e-3, random_state=42)
fcm_model.fit(sampled_pixels)
t_fcm = time.time() - t0

# Gán nhãn cho toàn bộ ảnh dựa vào tâm cụm FCM vừa tìm được
all_dists = np.linalg.norm(pixels_color[:, np.newaxis, :] - fcm_model.centers[np.newaxis, :, :], axis=2)
fcm_labels = np.argmin(all_dists, axis=1)
fcm_seg_img = fcm_model.centers[fcm_labels].reshape(h, w, 3).astype('uint8')

# Hiển thị kết quả FCM
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(img_color_np.astype('uint8'))
axes[0].set_title("Ảnh màu gốc", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(fcm_seg_img)
axes[1].set_title(f"Custom FCM (c = 5 cụm)\nThời gian: {t_fcm:.3f}s | FPC: {fcm_model.compute_fpc():.3f}", fontweight='bold', color='purple')
axes[1].axis('off')

plt.suptitle("KẾT QUẢ PHÂN ĐOẠN ẢNH BẰNG FUZZY C-MEANS (FCM)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Hoàn thành phân đoạn ảnh bằng FCM! Hệ số phân hoạch mờ FPC = {fcm_model.compute_fpc():.4f}")


---
# PHẦN 2: PHÂN CỤM TRÊN DỮ LIỆU BẢNG (DATASET) & BIỂU ĐỒ TẠI SAO CHỌN K, CHỌN C

### 2.1. Cơ sở lý thuyết: Tại sao và làm thế nào để chọn số cụm $K$ (K-Means) và $C$ (FCM)?
1. **Phương pháp Elbow (Khuỷu tay - WCSS/Inertia):**
   - Định nghĩa: WCSS (Within-Cluster Sum of Squares) là tổng bình phương khoảng cách từ mỗi điểm dữ liệu đến tâm cụm tương ứng:
     $$WCSS(K) = \sum_{k=1}^K \sum_{x \in C_k} ||x - \mu_k||^2$$
   - Nguyên lý: Khi $K$ tăng từ 1 lên, WCSS luôn giảm. Tại một giá trị $K^*$ nào đó, tốc độ giảm của WCSS đột ngột chậm lại, tạo thành một "điểm khuỷu tay" (Elbow point). Đó chính là số cụm tối ưu cân bằng giữa độ chính xác và độ phức tạp mô hình.

2. **Chỉ số Silhouette Score (Độ phân tách và độ kết cụm):**
   - Với mỗi điểm $i$, chỉ số silhouette là:
     $$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
     Trong đó:
     - $a(i)$: Khoảng cách trung bình từ $i$ đến các điểm khác trong cùng cụm (độ kết tụ nội cụm).
     - $b(i)$: Khoảng cách trung bình từ $i$ đến các điểm trong cụm gần nhất khác (độ phân tách liên cụm).
   - Điểm số $s(i) \in [-1, 1]$. Giá trị trung bình trên toàn tập dữ liệu càng gần 1 chứng tỏ các cụm càng cô đặc và cách xa nhau rõ ràng. **Giá trị $K$ (hoặc $C$) có Silhouette Score cao nhất là lựa chọn tối ưu.**

3. **Với Fuzzy C-Means (Hệ số phân hoạch mờ FPC):**
   $$FPC(c) = \frac{1}{N} \sum_{k=1}^N \sum_{i=1}^c u_{ik}^2$$
   - $FPC \in [1/c, 1]$. Khi $FPC \to 1$, phân hoạch mờ tiến gần đến phân hoạch cứng hoàn hảo (ít điểm bị phân vân giữa các cụm).


In [ ]:
# 2.2. NẠP DỮ LIỆU TỪ DATASET
# (LƯU Ý THI CỬ: Nếu thầy gửi file data.csv/data.xlsx, bạn mở comment dòng bên dưới để load file của thầy)
# df_custom = pd.read_csv('du_lieu_thay_gui.csv')
# X_data = df_custom.iloc[:, :-1].values # Bỏ cột nhãn nếu có

# Mặc định sử dụng dataset kinh điển Iris (hoặc Wine/Customer)
iris = load_iris()
X_raw = iris.data
y_true = iris.target
feature_names = iris.feature_names

# Chuẩn hóa dữ liệu (Standardization) để khoảng cách không bị chi phối bởi thang đo
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Đã nạp dataset gồm {X_scaled.shape[0]} mẫu, {X_scaled.shape[1]} đặc trưng.")
pd.DataFrame(X_raw, columns=feature_names).head()


In [ ]:
# 2.3. BIỂU ĐỒ CHỨNG MINH LÝ DO TẠI SAO CHỌN K VÀ CHỌN C (ELBOW & SILHOUETTE)

k_range = list(range(2, 9))
inertias = []
sil_kmeans = []
sil_fcm = []
fpcs = []

for k in k_range:
    # 1. Đo K-Means
    km = CustomKMeans(k=k, random_state=42)
    km_labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_kmeans.append(silhouette_score(X_scaled, km_labels))
    
    # 2. Đo FCM
    fcm = CustomFCM(c=k, random_state=42)
    fcm.fit(X_scaled)
    sil_fcm.append(silhouette_score(X_scaled, fcm.labels_))
    fpcs.append(fcm.compute_fpc())

best_k_sil = k_range[np.argmax(sil_kmeans)]
best_c_fcm = k_range[np.argmax(sil_fcm)]

# Vẽ 3 biểu đồ phân tích lựa chọn K và C
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Đồ thị 1: Elbow Method cho K-Means
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='Điểm khuỷu tay (Elbow) K=3')
axes[0].set_title("1. Phương pháp Elbow (Inertia/WCSS)", fontweight='bold')
axes[0].set_xlabel("Số cụm K")
axes[0].set_ylabel("Inertia (Tổng khoảng cách nội cụm)")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# Đồ thị 2: Silhouette Score cho K-Means
axes[1].plot(k_range, sil_kmeans, 'rs--', linewidth=2, markersize=8)
axes[1].scatter([best_k_sil], [max(sil_kmeans)], color='green', s=160, zorder=5, label=f'K tối ưu Silhouette = {best_k_sil}')
axes[1].set_title("2. Chỉ số Silhouette theo K (K-Means)", fontweight='bold')
axes[1].set_xlabel("Số cụm K")
axes[1].set_ylabel("Silhouette Score (càng cao càng tốt)")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

# Đồ thị 3: FPC & Silhouette cho Fuzzy C-Means
axes[2].plot(k_range, sil_fcm, 'gd-', linewidth=2, markersize=8, label='FCM Silhouette')
axes[2].plot(k_range, fpcs, 'm^--', linewidth=2, markersize=8, label='Hệ số mờ FPC')
axes[2].set_title("3. Đánh giá chọn C (Fuzzy C-Means)", fontweight='bold')
axes[2].set_xlabel("Số cụm C")
axes[2].set_ylabel("Điểm đánh giá")
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("BIỂU ĐỒ PHÂN TÍCH CHỌN SỐ CỤM K VÀ C TỐI ƯU", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"==> KẾT LUẬN CHỌN K VÀ C:")
print(f"1. Với K-Means: Đồ thị Elbow có điểm gãy (khuỷu tay) rõ rệt tại K = 3 (hoặc Silhouette cao nhất tại K = {best_k_sil}). Do đó ta chọn K = {best_k_sil}.")
print(f"2. Với FCM: Hệ số FPC và Silhouette đều đạt giá trị tối ưu tốt nhất tại C = {best_c_fcm}. Do đó chọn C = {best_c_fcm}.")


In [ ]:
# 2.4. SO SÁNH CHI TIẾT CUSTOM K-MEANS VS SKLEARN KMEANS TRÊN DATASET BẢNG
SELECTED_K = best_k_sil

# 1. Custom K-Means
t0 = time.time()
my_km = CustomKMeans(k=SELECTED_K, random_state=42)
my_km_labels = my_km.fit_predict(X_scaled)
t_my_km = time.time() - t0
my_km_sil = silhouette_score(X_scaled, my_km_labels)

# 2. Sklearn K-Means
t0 = time.time()
sk_km_tab = SklearnKMeans(n_clusters=SELECTED_K, random_state=42, n_init=10)
sk_km_labels = sk_km_tab.fit_predict(X_scaled)
t_sk_km = time.time() - t0
sk_km_sil = silhouette_score(X_scaled, sk_km_labels)

# Bảng đối sánh số liệu
df_tab_comp = pd.DataFrame({
    'Chỉ số đánh giá': [
        'Thuật toán / Thư viện',
        'Số cụm K được chọn',
        'Thời gian thực thi (giây)',
        'Inertia / WCSS',
        'Silhouette Score',
        'Số vòng lặp hội tụ'
    ],
    'Custom K-Means (Tự viết)': [
        'CustomKMeans (NumPy)',
        SELECTED_K,
        f"{t_my_km:.5f} s",
        f"{my_km.inertia_:.4f}",
        f"{my_km_sil:.4f}",
        my_km.n_iter_
    ],
    'Sklearn KMeans (Thư viện)': [
        'sklearn.cluster.KMeans',
        SELECTED_K,
        f"{t_sk_km:.5f} s",
        f"{sk_km_tab.inertia_:.4f}",
        f"{sk_km_sil:.4f}",
        sk_km_tab.n_iter_
    ]
})

print(f"--- BẢNG SO SÁNH PHÂN CỤM K-MEANS TRÊN DATASET (K={SELECTED_K}) ---")
display(df_tab_comp)

# Biểu đồ phân cụm 2D đối sánh Side-by-Side (Dùng 2 đặc trưng đầu tiên để trực quan)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=my_km_labels, cmap='viridis', s=60, edgecolors='k', alpha=0.8)
axes[0].scatter(my_km.centroids[:, 0], my_km.centroids[:, 1], c='red', s=200, marker='X', label='Tâm cụm Custom')
axes[0].set_title(f"Custom K-Means (K={SELECTED_K})\nSilhouette: {my_km_sil:.3f}", fontweight='bold', color='navy')
axes[0].set_xlabel("Đặc trưng 1 (Chuẩn hóa)")
axes[0].set_ylabel("Đặc trưng 2 (Chuẩn hóa)")
axes[0].legend()

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=sk_km_labels, cmap='viridis', s=60, edgecolors='k', alpha=0.8)
axes[1].scatter(sk_km_tab.cluster_centers_[:, 0], sk_km_tab.cluster_centers_[:, 1], c='red', s=200, marker='X', label='Tâm cụm Sklearn')
axes[1].set_title(f"Sklearn KMeans (K={SELECTED_K})\nSilhouette: {sk_km_sil:.3f}", fontweight='bold', color='darkgreen')
axes[1].set_xlabel("Đặc trưng 1 (Chuẩn hóa)")
axes[1].set_ylabel("Đặc trưng 2 (Chuẩn hóa)")
axes[1].legend()

plt.suptitle("ĐỐI SÁNH KẾT QUẢ PHÂN CỤM GIỮA CUSTOM VÀ THƯ VIỆN", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
# PHẦN 3: PHÂN LỚP BẰNG K-NEAREST NEIGHBORS (K-NN)

### 3.1. Cơ sở lý thuyết thuật toán K-NN & Cách lựa chọn $K$ tối ưu
1. **Nguyên lý K-NN (Instance-based / Lazy Learning):**
   - Thuật toán không có giai đoạn huấn luyện trừu tượng mà ghi nhớ dữ liệu.
   - Khi có điểm dữ liệu mới $x$:
     1. Tính khoảng cách từ $x$ tới tất cả các điểm trong tập huấn luyện (Euclidean hoặc Manhattan).
     2. Chọn ra $K$ điểm có khoảng cách nhỏ nhất (lân cận gần nhất).
     3. Tiến hành bỏ phiếu (Majority Voting): Điểm $x$ được gán nhãn của lớp chiếm đa số trong $K$ lân cận.
2. **Quy tắc và biểu đồ chọn $K$:**
   - **Nếu $K$ quá nhỏ ($K = 1$):** Mô hình rất nhạy với nhiễu (noise) và ngoại lai, dẫn đến hiện tượng **Overfitting** (quá khớp), biên phân chia ranh giới gồ ghề.
   - **Nếu $K$ quá lớn ($K 	o N$):** Ranh giới phân lớp bị làm phẳng quá mức, lớp chiếm đa số trong tập dữ liệu sẽ áp đảo các lớp thiểu số, dẫn đến **Underfitting** (thiếu khớp).
   - **Quy tắc chọn $K$ tối ưu:**
     - Nên chọn $K$ là số lẻ để tránh trường hợp hòa phiếu (tie).
     - Chia tập dữ liệu thành Train và Test/Validation. Vẽ đồ thị **Tỷ lệ lỗi (Error Rate)** hoặc **Độ chính xác (Accuracy)** theo các giá trị $K = 1, 3, 5, \dots, 21$.
     - Chọn giá trị $K$ có tỷ lệ lỗi thấp nhất và độ ổn định cao nhất.


In [ ]:
# 3.2. CÀI ĐẶT THUẬT TOÁN K-NN TỪ ĐẦU (CUSTOM KNN FROM SCRATCH)
class CustomKNN:
    """
    Thuật toán K-Nearest Neighbors tự cài đặt 100% bằng NumPy.
    - Hỗ trợ khoảng cách: 'euclidean', 'manhattan'.
    - Bỏ phiếu theo đa số (Majority Vote) kèm xử lý tie-break bằng tổng khoảng cách.
    """
    def __init__(self, k=5, metric='euclidean'):
        self.k = k
        self.metric = metric
        self.X_train = None
        self.y_train = None
        self.classes_ = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        self.classes_ = np.unique(self.y_train)
        return self

    def _compute_distance(self, x):
        if self.metric == 'euclidean':
            return np.linalg.norm(self.X_train - x, axis=1)
        elif self.metric == 'manhattan':
            return np.sum(np.abs(self.X_train - x), axis=1)
        else:
            raise ValueError("Metric phải là 'euclidean' hoặc 'manhattan'")

    def predict_one(self, x):
        dists = self._compute_distance(x)
        # Lấy chỉ số của k lân cận gần nhất
        k_indices = np.argsort(dists)[:self.k]
        k_nearest_labels = self.y_train[k_indices]
        k_nearest_dists = dists[k_indices]

        # Đếm số phiếu cho từng nhãn
        labels, counts = np.unique(k_nearest_labels, return_counts=True)
        max_count = np.max(counts)
        candidates = labels[counts == max_count]

        # Xử lý trường hợp hòa phiếu (Tie-break): chọn lớp có tổng khoảng cách nhỏ nhất
        if len(candidates) == 1:
            return candidates[0]
        else:
            cand_dist_sum = {c: np.sum(k_nearest_dists[k_nearest_labels == c]) for c in candidates}
            return min(cand_dist_sum, key=cand_dist_sum.get)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self.predict_one(x) for x in X])

    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == np.asarray(y))

print("Đã định nghĩa class CustomKNN thành công!")


In [ ]:
# 3.3. CHIA DỮ LIỆU & VẼ BIỂU ĐỒ TẠI SAO CHỌN K CHO K-NN
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_true, test_size=0.3, random_state=42, stratify=y_true)

knn_k_range = list(range(1, 22, 2)) # Thử nghiệm các giá trị K lẻ từ 1 đến 21
knn_errors_euc = []
knn_errors_man = []
knn_accs_euc = []

for k in knn_k_range:
    # 1. Đo với khoảng cách Euclidean
    knn_euc = CustomKNN(k=k, metric='euclidean')
    knn_euc.fit(X_train, y_train)
    acc_euc = knn_euc.score(X_test, y_test)
    knn_accs_euc.append(acc_euc)
    knn_errors_euc.append(1.0 - acc_euc)
    
    # 2. Đo với khoảng cách Manhattan
    knn_man = CustomKNN(k=k, metric='manhattan')
    knn_man.fit(X_train, y_train)
    acc_man = knn_man.score(X_test, y_test)
    knn_errors_man.append(1.0 - acc_man)

best_knn_k = knn_k_range[np.argmin(knn_errors_euc)]

# Vẽ biểu đồ tỷ lệ lỗi theo K
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(knn_k_range, knn_errors_euc, 'ro-', linewidth=2, markersize=7, label='Euclidean Distance')
axes[0].plot(knn_k_range, knn_errors_man, 'bs--', linewidth=2, markersize=7, label='Manhattan Distance')
axes[0].scatter([best_knn_k], [min(knn_errors_euc)], color='green', s=160, zorder=5, label=f'K tối ưu = {best_knn_k}')
axes[0].set_title("Tỷ lệ lỗi (Error Rate) theo K", fontweight='bold')
axes[0].set_xlabel("Số lân cận K")
axes[0].set_ylabel("Tỷ lệ lỗi (Error Rate)")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(knn_k_range, [acc * 100 for acc in knn_accs_euc], 'go-', linewidth=2, markersize=7, label='Accuracy (%)')
axes[1].set_title("Độ chính xác (Accuracy %) theo K", fontweight='bold')
axes[1].set_xlabel("Số lân cận K")
axes[1].set_ylabel("Độ chính xác (%)")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle("BIỂU ĐỒ CHỨNG MINH LỰA CHỌN K TỐI ƯU CHO K-NN", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"==> KẾT LUẬN CHỌN K CHO K-NN:")
print(f"Dựa vào biểu đồ trên, giá trị K = {best_knn_k} cho tỷ lệ lỗi thấp nhất ({min(knn_errors_euc)*100:.2f}%) và độ chính xác đạt cực đại ({max(knn_accs_euc)*100:.2f}%).")


In [ ]:
# 3.4. SO SÁNH CUSTOM K-NN VS SKLEARN KNEIGHBORSCLASSIFIER
SELECTED_KNN_K = best_knn_k

# 1. Custom KNN
t0 = time.time()
custom_knn = CustomKNN(k=SELECTED_KNN_K, metric='euclidean')
custom_knn.fit(X_train, y_train)
custom_preds = custom_knn.predict(X_test)
t_custom_knn = time.time() - t0
custom_knn_acc = accuracy_score(y_test, custom_preds)

# 2. Sklearn KNN
t0 = time.time()
sk_knn_model = SklearnKNN(n_neighbors=SELECTED_KNN_K, metric='euclidean')
sk_knn_model.fit(X_train, y_train)
sk_preds = sk_knn_model.predict(X_test)
t_sk_knn = time.time() - t0
sk_knn_acc = accuracy_score(y_test, sk_preds)

# So sánh độ khớp giữa Custom và Thư viện
match_rate = np.mean(custom_preds == sk_preds) * 100

# Bảng so sánh
df_knn_comp = pd.DataFrame({
    'Tiêu chí so sánh': [
        'Tên mô hình',
        'Số lân cận K',
        'Độ đo khoảng cách',
        'Thời gian dự đoán (giây)',
        'Độ chính xác (Accuracy)',
        'Độ tương đồng Custom vs Sklearn'
    ],
    'Custom KNN (Tự viết)': [
        'CustomKNN (NumPy)',
        SELECTED_KNN_K,
        'Euclidean',
        f"{t_custom_knn:.5f} s",
        f"{custom_knn_acc * 100:.2f}%",
        "100.00% (Khớp hoàn toàn)" if match_rate == 100 else f"{match_rate:.2f}%"
    ],
    'Sklearn KNeighborsClassifier': [
        'sklearn.neighbors.KNeighborsClassifier',
        SELECTED_KNN_K,
        'Euclidean',
        f"{t_sk_knn:.5f} s",
        f"{sk_knn_acc * 100:.2f}%",
        f"{match_rate:.2f}%"
    ]
})

print(f"--- BẢNG SO SÁNH PHÂN LỚP K-NN TRÊN TẬP TEST (K={SELECTED_KNN_K}) ---")
display(df_knn_comp)

# Ma trận nhầm lẫn (Confusion Matrix) side-by-side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cm_custom = confusion_matrix(y_test, custom_preds)
cm_sk = confusion_matrix(y_test, sk_preds)

im1 = axes[0].imshow(cm_custom, cmap='Blues')
axes[0].set_title(f"Confusion Matrix (Custom KNN)\nAcc: {custom_knn_acc*100:.1f}%", fontweight='bold')
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")
for i in range(cm_custom.shape[0]):
    for j in range(cm_custom.shape[1]):
        axes[0].text(j, i, cm_custom[i, j], ha="center", va="center", color="red" if cm_custom[i, j] > 0 else "black", fontweight='bold')

im2 = axes[1].imshow(cm_sk, cmap='Greens')
axes[1].set_title(f"Confusion Matrix (Sklearn KNN)\nAcc: {sk_knn_acc*100:.1f}%", fontweight='bold')
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")
for i in range(cm_sk.shape[0]):
    for j in range(cm_sk.shape[1]):
        axes[1].text(j, i, cm_sk[i, j], ha="center", va="center", color="red" if cm_sk[i, j] > 0 else "black", fontweight='bold')

plt.suptitle("MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ĐỐI SÁNH", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## TỔNG KẾT BÀI THI / ÔN TẬP
1. **Bài toán phân đoạn ảnh:** Đã thực thi phân đoạn trên cả ảnh màu RGB và ảnh trắng đen Grayscale với $K = 5$ và $K = 7$ cụm, chứng minh thuật toán Custom hội tụ chuẩn xác và cho kết quả tương đương thư viện Scikit-Learn.
2. **Chọn $K$ và $C$:** Đã trực quan hóa bằng phương pháp Elbow, Silhouette Score và hệ số FPC, cung cấp căn cứ toán học vững chắc cho việc lựa chọn số cụm tối ưu.
3. **Phân lớp K-NN:** Đã xây dựng hàm Custom K-NN đạt độ chính xác tương đương 100% với thư viện Scikit-Learn và chỉ ra $K$ tối ưu dựa trên đường cong tỷ lệ lỗi.
